In [4]:
# ══════════════════════════════════════════════════════════════════════════════
# SITE DIAGNOSTICS — Standalone Script
# Checks for site/scanner differences before harmonisation.
# Outputs: console summary + PNG plots saved to OUTPUT_DIR
# ══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import nibabel as nib
from scipy.stats import kruskal, chi2_contingency, mannwhitneyu
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# ── Configuration ─────────────────────────────────────────────────────────────
BIDS_ROOT      = r"..\ADHD_BIDS"        # Same as your main script
LABEL_COLUMN   = "label"             # Column name for ADHD/control label
OUTPUT_DIR     = "..\output\site_diagnostics"  # Folder to save plots and CSVs
MAX_SUBJECTS   = None                # Set to e.g. 100 to subsample for speed. None = all.
# ─────────────────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
WARN  = []   # collect warnings to print at the end
PASS  = []   # collect passing checks

def flag(label, condition, detail=""):
    """Print a pass/warn line and record it."""
    if condition:
        msg = f"  ⚠  WARN  | {label}"
    else:
        msg = f"  ✓  PASS  | {label}"
    if detail:
        msg += f"  →  {detail}"
    print(msg)
    (WARN if condition else PASS).append(label)

In [5]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── set up a text logfile and tee stdout/stderr ────────────────────────────────
log_path = os.path.join(OUTPUT_DIR, "diagnostics.txt")
import sys
class Tee:
    def __init__(self, *streams):
        self.streams = streams
    def write(self, data):
        for s in self.streams:
            try:
                s.write(data)
            except UnicodeEncodeError:
                # some streams (e.g. the notebook's) can't handle box‑drawing chars;
                # replace unencodable bytes rather than crash.
                s.write(data.encode("ascii", "replace").decode("ascii"))
            s.flush()
    def flush(self):
        for s in self.streams:
            try:
                s.flush()
            except Exception:
                pass

_log_file = open(log_path, "w", encoding="utf-8")
sys.stdout = Tee(sys.stdout, _log_file)
sys.stderr = Tee(sys.stderr, _log_file)

In [6]:
# ══════════════════════════════════════════════════════════════════════════════
# 0. Load participants.tsv
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print("  SITE DIAGNOSTICS")
print("═"*70)

tsv_path = os.path.join(BIDS_ROOT, "participants.tsv")
if not os.path.exists(tsv_path):
    raise FileNotFoundError(f"participants.tsv not found at {tsv_path}")

participants = pd.read_csv(tsv_path, sep="\t")
participants = participants[participants[LABEL_COLUMN].notnull()].copy()
participants[LABEL_COLUMN] = participants[LABEL_COLUMN].astype(int)

if "site" not in participants.columns:
    raise ValueError("No 'site' column found in participants.tsv. Cannot run site diagnostics.")

sites = sorted(participants["site"].unique())
print(f"\nLoaded {len(participants)} subjects across {len(sites)} sites: {sites}\n")

if MAX_SUBJECTS:
    participants = participants.sample(n=min(MAX_SUBJECTS, len(participants)), random_state=42)
    print(f"  [Subsampled to {len(participants)} subjects for speed]\n")

In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# 1. Class Balance per Site
# ══════════════════════════════════════════════════════════════════════════════
print("─"*70)
print("CHECK 1 — Class Balance per Site")
print("─"*70)

balance = participants.groupby("site")[LABEL_COLUMN].agg(
    n="count",
    n_adhd="sum",
    adhd_rate="mean"
).reset_index()
balance["n_control"] = balance["n"] - balance["n_adhd"]
balance["adhd_rate"] = balance["adhd_rate"].round(3)

print(balance.to_string(index=False))

rate_range = balance["adhd_rate"].max() - balance["adhd_rate"].min()
flag("ADHD prevalence consistent across sites",
     condition=rate_range > 0.15,
     detail=f"range = {rate_range:.2f} (>{0.15} is concerning)")

# Save CSV
balance.to_csv(os.path.join(OUTPUT_DIR, "class_balance_by_site.csv"), index=False)

# Plot
fig, ax = plt.subplots(figsize=(max(6, len(sites)*1.5), 5), facecolor="white")
x = np.arange(len(balance))
w = 0.35
ax.bar(x - w/2, balance["n_control"], w, label="Control", color="#4C72B0", alpha=0.85)
ax.bar(x + w/2, balance["n_adhd"],    w, label="ADHD",    color="#DD8452", alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(balance["site"], rotation=25, ha="right")
ax.set_ylabel("Subject Count"); ax.set_title("Class Balance per Site")
ax.legend(); fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "1_class_balance.png"), dpi=150)
plt.close(fig)

In [8]:
# ══════════════════════════════════════════════════════════════════════════════
# 2. Demographic Differences (Age & Sex)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("CHECK 2 — Demographic Differences (Age & Sex)")
print("─"*70)

has_age = "age" in participants.columns
has_sex = "gender" in participants.columns

if has_age:
    age_groups = [g["age"].dropna().values for _, g in participants.groupby("site")]
    age_groups  = [g for g in age_groups if len(g) > 1]
    if len(age_groups) >= 2:
        stat, p_age = kruskal(*age_groups)
        print(f"\n  Age — Kruskal-Wallis H={stat:.2f}, p={p_age:.4f}")
        flag("Age is consistent across sites", condition=p_age < 0.05,
             detail=f"p={p_age:.4f} (<0.05 = significant difference)")

        age_summary = participants.groupby("site")["age"].agg(["mean","std","median"]).round(2)
        print(age_summary.to_string())

        # Age boxplot
        fig, ax = plt.subplots(figsize=(max(6, len(sites)*1.5), 5), facecolor="white")
        data_plot = [participants[participants["site"]==s]["age"].dropna().values for s in sites]
        bp = ax.boxplot(data_plot, labels=sites, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("#4C72B0"); patch.set_alpha(0.7)
        ax.set_title(f"Age Distribution by Site  (Kruskal-Wallis p={p_age:.4f})")
        ax.set_ylabel("Age"); ax.set_xlabel("Site")
        plt.xticks(rotation=25, ha="right")
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, "2a_age_by_site.png"), dpi=150)
        plt.close(fig)
else:
    print("  [SKIP] No 'age' column found.")

if has_sex:
    ct = pd.crosstab(participants["site"], participants["gender"])
    print(f"\n  Sex counts:\n{ct.to_string()}")
    if ct.shape[1] >= 2 and ct.shape[0] >= 2:
        chi2, p_sex, _, _ = chi2_contingency(ct)
        print(f"\n  Sex — Chi² = {chi2:.2f}, p = {p_sex:.4f}")
        flag("Sex ratio is consistent across sites", condition=p_sex < 0.05,
             detail=f"p={p_sex:.4f} (<0.05 = significant difference)")

        # Sex stacked bar
        sex_pct = ct.div(ct.sum(axis=1), axis=0)
        fig, ax = plt.subplots(figsize=(max(6, len(sites)*1.5), 5), facecolor="white")
        sex_pct.plot(kind="bar", stacked=True, ax=ax, colormap="Set2", alpha=0.85)
        ax.set_title(f"Sex Ratio by Site  (Chi² p={p_sex:.4f})")
        ax.set_ylabel("Proportion"); ax.set_xlabel("Site")
        plt.xticks(rotation=25, ha="right")
        ax.legend(title="Sex", bbox_to_anchor=(1.01, 1), loc="upper left")
        fig.tight_layout()
        fig.savefig(os.path.join(OUTPUT_DIR, "2b_sex_by_site.png"), dpi=150)
        plt.close(fig)
else:
    print("  [SKIP] No 'sex' column found.")

In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# 3. MRI Intensity Distribution per Site
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("CHECK 3 — MRI Intensity Distributions")
print("─"*70)
print("  Loading MRI files (this may take a few minutes)...")

intensity_records = []
missing = 0
for _, row in participants.iterrows():
    sub_id = str(row["participant_id"])
    path   = os.path.join(BIDS_ROOT, f"sub-{sub_id}", "anat", f"{sub_id}_T1w.nii.gz")
    if not os.path.exists(path):
        missing += 1
        continue
    try:
        data  = nib.load(path).get_fdata()
        brain = data[data > data.mean() * 0.1]   # rough brain mask: above 10% of mean
        intensity_records.append({
            "sub_id":  sub_id,
            "site":    row["site"],
            "label":   row[LABEL_COLUMN],
            "mean":    float(brain.mean()),
            "std":     float(brain.std()),
            "median":  float(np.median(brain)),
            "p25":     float(np.percentile(brain, 25)),
            "p75":     float(np.percentile(brain, 75)),
        })
    except Exception as e:
        print(f"  [WARNING] Could not load sub-{sub_id}: {e}")

if missing:
    print(f"  [WARNING] {missing} MRI files not found — skipped.")

df_int = pd.DataFrame(intensity_records)
df_int.to_csv(os.path.join(OUTPUT_DIR, "intensity_stats_by_subject.csv"), index=False)

if df_int.empty:
    print("  [ERROR] No MRI files loaded. Skipping intensity checks.")
else:
    int_summary = df_int.groupby("site")[["mean","std","median"]].describe().round(2)
    print(f"\n  Intensity summary:\n{int_summary.to_string()}")

    # Kruskal-Wallis on mean intensity across sites
    int_groups = [g["mean"].values for _, g in df_int.groupby("site") if len(g) > 1]
    if len(int_groups) >= 2:
        stat, p_int = kruskal(*int_groups)
        print(f"\n  Mean intensity — Kruskal-Wallis H={stat:.2f}, p={p_int:.4f}")
        flag("MRI mean intensity consistent across sites", condition=p_int < 0.05,
             detail=f"p={p_int:.4f}")

    # Boxplot: mean intensity per site
    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor="white")
    for ax, col, title in zip(axes, ["mean","std","median"],
                               ["Mean Intensity","Std Intensity","Median Intensity"]):
        data_plot = [df_int[df_int["site"]==s][col].values for s in sites]
        bp = ax.boxplot(data_plot, labels=sites, patch_artist=True)
        for patch in bp["boxes"]:
            patch.set_facecolor("#55A868"); patch.set_alpha(0.7)
        ax.set_title(title); ax.set_xlabel("Site")
        plt.sca(ax); plt.xticks(rotation=25, ha="right")
    fig.suptitle("MRI Intensity Statistics by Site", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "3_intensity_by_site.png"), dpi=150)
    plt.close(fig)

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# 4. PCA — does data cluster by site or by label?
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("CHECK 4 — PCA (Site Separation vs Label Separation)")
print("─"*70)

if df_int.empty:
    print("  [SKIP] No intensity data loaded.")
else:
    feature_cols = ["mean", "std", "median", "p25", "p75"]
    X = df_int[feature_cols].values
    X_scaled = StandardScaler().fit_transform(X)
    pcs = PCA(n_components=2, random_state=42).fit_transform(X_scaled)

    df_int["PC1"] = pcs[:, 0]
    df_int["PC2"] = pcs[:, 1]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor="white")
    cmap_sites = plt.cm.get_cmap("tab10", len(sites))
    cmap_label = {0: "#4C72B0", 1: "#DD8452"}

    # Left: coloured by site
    ax = axes[0]
    for i, site in enumerate(sites):
        mask = df_int["site"] == site
        ax.scatter(df_int.loc[mask, "PC1"], df_int.loc[mask, "PC2"],
                   label=site, color=cmap_sites(i), alpha=0.7, s=40)
    ax.set_title("PCA — coloured by Site\n(clusters here = scanner bias)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(fontsize=8, title="Site", bbox_to_anchor=(1.01,1), loc="upper left")

    # Right: coloured by label
    ax = axes[1]
    for lbl, lbl_name in [(0, "Control"), (1, "ADHD")]:
        mask = df_int["label"] == lbl
        ax.scatter(df_int.loc[mask, "PC1"], df_int.loc[mask, "PC2"],
                   label=lbl_name, color=cmap_label[lbl], alpha=0.7, s=40)
    ax.set_title("PCA — coloured by Label\n(clusters here = real signal)")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    ax.legend(fontsize=8, title="Label")

    fig.suptitle("PCA of MRI Intensity Features", fontsize=13, fontweight="bold")
    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_DIR, "4_pca.png"), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print("  PCA plot saved. Inspect: do subjects cluster by site or by label?")

In [11]:
# ══════════════════════════════════════════════════════════════════════════════
# 5. Label–Site Confound (does ADHD co-vary with site?)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "─"*70)
print("CHECK 5 — Label–Site Confound")
print("─"*70)

ct_label_site = pd.crosstab(participants["site"], participants[LABEL_COLUMN])
print(ct_label_site.to_string())
if ct_label_site.shape[1] >= 2 and ct_label_site.shape[0] >= 2:
    chi2_ls, p_ls, _, _ = chi2_contingency(ct_label_site)
    print(f"\n  Chi² = {chi2_ls:.2f}, p = {p_ls:.4f}")
    flag("Label distribution is independent of site", condition=p_ls < 0.05,
         detail=f"p={p_ls:.4f}  (<0.05 means label & site are confounded)")
else:
    print("  [SKIP] Not enough sites or labels for chi² test.")

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*70)
print("  SUMMARY")
print("═"*70)
print(f"\n  {len(PASS)} checks passed, {len(WARN)} warnings raised.\n")
if WARN:
    print("  Warnings (address before harmonisation):")
    for w in WARN:
        print(f"    ⚠  {w}")
if PASS:
    print("\n  Passed:")
    for p in PASS:
        print(f"    ✓  {p}")

print(f"\n  Plots and CSVs saved to: ./{OUTPUT_DIR}/")
print("═"*70 + "\n")

# tidy up – close the log file if it exists
try:
    _log_file.close()
except NameError:
    pass